# v122 — cons2of3_c26 consensus 提出 CSV 生成 (Colab、verified のみ)

**前提**: `solar_filament_v121/work/latest/v121_summary.json` の
`gate.decision == 'cross_fold_verified'` を実行前に確認 —
fold-0 で Δ@0.45 >= +0.005 が再現された場合のみ提出物を生成する
(§2.99 事前登録プロトコルの機械的強制)。未検証なら中断します。

パイプライン = prod 提出と同一、detector 段のみ置換:
- 3 det (prod yolo_fold_2 + v116 crop + v117 yolo26、sha pin 検証)
  を floor 0.08 で推論 → mask-IoU>=0.5 クラスタ → **conf>=0.26 の
  支持が >=2 det** のクラスタのみ放出 (cons2of3_c26)
- 以降 prod verbatim: edgeatt all-data refiner → threshold 0.45 →
  ownership drop_contained 0.90 → min_area 20 → RLE → CSV

出力: Drive `artifacts/consensus_v122/submission_consensus.csv`
+ `audit_consensus.json` + `efficiency_consensus.json`。
提出はユーザー判断 — CSV 生成は提出そのものではありません。


In [ ]:
# [1/4] Drive mount + 依存 + v121c verified ガード + パス検証
from google.colab import drive
drive.mount('/content/drive')

%pip install -q 'ultralytics>=8.3,<9' pycocotools

import json, shutil, zipfile
from pathlib import Path
DRIVE = Path('/content/drive/MyDrive')
V30 = DRIVE / 'solar_filament_v30'

# §2.99 — fold-0 複製が cross_fold_verified でなければ提出 CSV を生成しない
summary_path = DRIVE / 'solar_filament_v121' / 'work' / 'latest' / 'v121_summary.json'
if not summary_path.exists():
    raise RuntimeError(
        f'v121c summary がありません: {summary_path} — fold-0 eval を先に実行してください')
gate = json.loads(summary_path.read_text(encoding='utf-8'))['gate']
print('v121c gate:', json.dumps(gate, indent=1)[:600])
if gate.get('decision') != 'cross_fold_verified':
    raise RuntimeError(
        f"fold-0 複製が未検証 ({gate.get('decision')}) — consensus 提出は生成しません (§2.99)")
print('cross_fold_verified — consensus 提出生成に進みます')

# repro_code の解決 (prod repro と同一規約)
if (V30 / 'repro_code' / 'scratch' / 'run_test_submission.py').exists():
    REPRO = V30 / 'repro_code'
    print('repro_code (drive dir):', REPRO)
else:
    zip_path = V30 / 'repro_code.zip'
    assert zip_path.exists(), 'repro_code がありません'
    REPRO = Path('/content/repro_code')
    if not (REPRO / 'scratch' / 'run_test_submission.py').exists():
        REPRO.mkdir(exist_ok=True)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(REPRO)
    print('repro_code (unzipped):', REPRO)

checks = {
    'test_images': V30 / 'competition_data' / 'MAGFiLO_1.0_Kaggle_2026' / 'test' / 'test_images',
    'yolo_fold2': V30 / 'runs' / 'yolo_fold_2' / 'weights' / 'best.pt',
    'crop_f2': V30 / 'runs' / 'v116_croptrain' / 'yolo_croptrain_fold2' / 'weights' / 'best.pt',
    'yolo26_f2': V30 / 'runs' / 'v117_yolo26' / 'yolo_yolo26m_fold2' / 'weights' / 'best.pt',
    'refiner': V30 / 'artifacts' / 'mediseg_edgeattnet_refiner_all_data_resume.pt',
}
for label, p in checks.items():
    print(label, p.exists(), p)
    assert p.exists(), f'{label} がありません: {p}'
print('n_test:', len(list(checks['test_images'].glob('*.jpeg'))))
print('全パスOK')


In [ ]:
# [2/4] consensus submission script を repro_code/scratch へ書込み
CONSENSUS_SUB = '"""test 180画像の推論 → consensus (cons2of3_c26) submission.csv (contingent v122)。\n\n**実行条件**: v121c fold-0 eval が `cross_fold_verified`\n(cons2of3_c26 Δ@0.45 >= +0.005, §2.99) を出した場合のみ実行する。\nunverified ならこのスクリプトは提出物を作らない (検証前の提出は\n事前登録プロトコルに反する)。\n\n`run_test_submission.py --mode prod` からの派生。差分は提案生成のみ:\nprod の単一 YOLO を、§2.98 で検証した 3 detector コンセンサスに\n置き換える。後続 (refine_crops → threshold → ownership → RLE → CSV)\nは一切変更しない。\n\n提案生成 (§2.98 で計測したものと同一 semantics):\n  - detector: prod yolo11m fold_2 / v116 croptrain fold_2 /\n    v117 yolo26m fold_2 (全て folds(0,1) 学習 — test は完全 disjoint\n    なので fold-2 学習済みの使用は正当。all-data 版は未計測構成の\n    ため、計測済み構成のまま提出する)\n  - 各 det を conf floor 0.08・imgsz=1280・iou=0.55・agnostic_nms・\n    retina_masks で推論 (prod 推論パラメータと同一)\n  - mask-IoU>=0.5 の推移クラスタ (union-find、cross-detector edge のみ)\n  - 放出 = 支持 member のうち conf>=0.26 が >=2 detector のクラスタ、\n    max-conf member の mask + conf\n\n使い方:\n    python scratch/run_test_submission_consensus.py --device cuda \\\n        --out output/test_submission_consensus\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nimport time\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom pycocotools import mask as mask_util\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT))\nsys.path.insert(0, str(ROOT / "scratch"))\n\nfrom preprocess_pseudo_label_images import solar_three_channel  # noqa: E402\nfrom run_refiner_pq_allfold import refine_crops, square_box  # noqa: E402\nfrom solar_filament.pipeline.io_images import (  # noqa: E402\n    imread_unicode_with_retry,\n)\nfrom solar_filament.pipeline.ownership import apply_ownership  # noqa: E402\nfrom solar_filament.pipeline.paths import (  # noqa: E402\n    artifacts_dir,\n    resolve_drive,\n)\nfrom src.mediseg_models import MediSegEdgeAttUNet  # noqa: E402\n\n# --- 本番確定値 (run_test_submission.py prod と同一) -------------------------\nYOLO_IMGSZ = 1280\nPREDICT_IOU = 0.55\nREFINER_THRESHOLD = 0.45\nDROP_THRESHOLD = 0.90\nMIN_AREA = 20\n\n# --- §2.98 固定パラメータ -----------------------------------------------------\nDET_FLOOR = 0.08          # 全 detector の提案保持 floor\nCONSENSUS_IOU = 0.5       # cross-detector cluster の mask-IoU 閾値\nSUPPORT_CONF = 0.26       # cons2of3_c26 の「支持」とみなす member conf\n\nREFINER_FILENAME = "mediseg_edgeattnet_refiner_all_data_resume.pt"  # prod 同一\n\n# 計測済み fold-2 detector セット + sha256 pin (§2.98 記録値)\nCONSENSUS_DETECTORS = {\n    "yolo": ("runs/yolo_fold_2/weights/best.pt",\n             "40ec31fd2c6aa8e32c952e64be453f4351ace339d31ef747b4f28624eafe7e8c"),\n    "crop": ("runs/v116_croptrain/yolo_croptrain_fold2/weights/best.pt",\n             "c23b1bb79620c62fd85984d032310bae26c240fd1be0aaf7df364fbc560fbcda"),\n    "yolo26": ("runs/v117_yolo26/yolo_yolo26m_fold2/weights/best.pt",\n               "0c8bb71f6ab4398aaea2b729992a3b88f3f4d707e0cbcbe35f1b82a2533db7a5"),\n}\n\n\ndef sha256_of(path: Path) -> str:\n    digest = hashlib.sha256()\n    with open(path, "rb") as handle:\n        for chunk in iter(lambda: handle.read(1 << 20), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef result_masks(result) -> tuple[list[np.ndarray], np.ndarray]:\n    if result.masks is None:\n        return [], np.zeros(0, np.float32)\n    output = [m >= 0.5 for m in result.masks.data.detach().cpu().numpy()]\n    confidences = (\n        result.boxes.conf.detach().cpu().numpy().astype(np.float32)\n        if result.boxes is not None else np.zeros(0, np.float32))\n    return output, confidences\n\n\ndef _mask_iou(a: np.ndarray, b: np.ndarray) -> float:\n    inter = int(np.logical_and(a, b).sum())\n    return inter / int(np.logical_or(a, b).sum()) if inter else 0.0\n\n\ndef consensus_clusters(props, iou_thr):\n    """run_fold_oof_eval_consensus.py と同一: cross-detector pair のみ\n    mask-IoU>=iou_thr で結合する union-find (同 detector 内 edge なし)。"""\n    n = len(props)\n    parent = list(range(n))\n\n    def find(x):\n        while parent[x] != x:\n            parent[x] = parent[parent[x]]\n            x = parent[x]\n        return x\n    boxes = []\n    for _, m, _ in props:\n        ys, xs = np.nonzero(m)\n        boxes.append((int(xs.min()), int(ys.min()),\n                      int(xs.max()) + 1, int(ys.max()) + 1) if xs.size\n                     else (0, 0, 0, 0))\n    areas = [int(m.sum()) for _, m, _ in props]\n    for i in range(n):\n        ax0, ay0, ax1, ay1 = boxes[i]\n        for j in range(i + 1, n):\n            if props[i][0] == props[j][0]:\n                continue\n            bx0, by0, bx1, by1 = boxes[j]\n            inter = (max(0, min(ax1, bx1) - max(ax0, bx0))\n                     * max(0, min(ay1, by1) - max(ay0, by0)))\n            if inter < iou_thr * max(areas[i], areas[j]):\n                continue\n            if _mask_iou(props[i][1], props[j][1]) >= iou_thr:\n                ri, rj = find(i), find(j)\n                if ri != rj:\n                    parent[ri] = rj\n    groups = defaultdict(list)\n    for i in range(n):\n        groups[find(i)].append(i)\n    return list(groups.values())\n\n\ndef emit_cons2of3_c26(props):\n    """§2.98 arm と同一: conf>=SUPPORT_CONF の支持が >=2 detector の\n    クラスタから max-conf member を放出 → (mask, conf) リスト。"""\n    emitted = []\n    for mem in consensus_clusters(props, CONSENSUS_IOU):\n        strong = [i for i in mem if props[i][2] >= SUPPORT_CONF]\n        if len({props[i][0] for i in strong}) >= 2:\n            best = max(mem, key=lambda i: props[i][2])\n            emitted.append((props[best][1], props[best][2]))\n    return emitted\n\n\ndef _run_submission(local: Path) -> int:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--drive", type=Path, default=None)\n    parser.add_argument("--limit", type=int, default=0)\n    parser.add_argument("--threads", type=int, default=14)\n    parser.add_argument("--device", type=str, default="cpu")\n    parser.add_argument("--out", type=Path,\n                        default=Path("output/test_submission_consensus"))\n    args = parser.parse_args()\n    torch.set_num_threads(args.threads)\n    args.out.mkdir(parents=True, exist_ok=True)\n\n    drive = resolve_drive(args.drive)\n    artifacts = artifacts_dir(drive)\n    test_dir = (drive / "solar_filament_v30" / "competition_data"\n                / "MAGFiLO_1.0_Kaggle_2026" / "test" / "test_images")\n    test_files = sorted(p.name for p in test_dir.glob("*.jpeg"))\n    assert len(test_files) == 180, f"test画像が180枚ではありません: {len(test_files)}"\n    if args.limit:\n        test_files = test_files[:args.limit]\n\n    local_img = local / "images"\n    local_w = local / "weights"\n    for d in (local_img, local_w):\n        d.mkdir(parents=True, exist_ok=True)\n\n    # --- detector staging + sha256 pin 検証 (§2.98 記録値) --------------------\n    det_paths = {}\n    for det_name, (rel, expected_sha) in CONSENSUS_DETECTORS.items():\n        src = drive / "solar_filament_v30" / rel\n        if not src.exists():\n            raise FileNotFoundError(f"detector weights missing: {src}")\n        actual = sha256_of(src)\n        if actual != expected_sha:\n            raise RuntimeError(\n                f"detector {det_name} sha256 mismatch: {actual[:16]} != "\n                f"{expected_sha[:16]} (§2.98 pin)")\n        dst = local_w / f"{det_name}.pt"\n        print(f"copy {src.name} ({det_name}) -> {dst}  sha={actual[:12]}",\n              flush=True)\n        shutil.copy2(src, dst)\n        det_paths[det_name] = dst\n    refiner_src = artifacts / REFINER_FILENAME\n    refiner_path = local_w / "refiner_edgeatt_all.pt"\n    print(f"copy {refiner_src.name} -> {refiner_path}", flush=True)\n    shutil.copy2(refiner_src, refiner_path)\n\n    # --- 前処理 (solar_three_channel、prod と同一) ----------------------------\n    for name in test_files:\n        dst = local_img / name\n        gray = None\n        for attempt in range(4):\n            raw = None\n            try:\n                raw = np.frombuffer((test_dir / name).read_bytes(), np.uint8)\n            except OSError:\n                time.sleep(2.0 * (attempt + 1))\n                continue\n            gray = cv2.imdecode(raw, cv2.IMREAD_GRAYSCALE)\n            if gray is not None:\n                break\n            time.sleep(2.0 * (attempt + 1))\n        if gray is None:\n            raise FileNotFoundError(f"test画像を読めません: {name}")\n        cv2.imwrite(str(dst), solar_three_channel(gray),\n                    [cv2.IMWRITE_JPEG_QUALITY, 96])\n    print({"test画像": len(test_files), "stage": str(local_img)}, flush=True)\n\n    from ultralytics import YOLO\n    det_models = {n: YOLO(str(p)) for n, p in det_paths.items()}\n    refiner = MediSegEdgeAttUNet(3, 1, False).eval().to(args.device)\n    checkpoint = torch.load(refiner_path, map_location="cpu")\n    refiner.load_state_dict(\n        checkpoint.get("best_model") or checkpoint["model"], strict=True)\n\n    rows, audit = [], []\n    started = time.perf_counter()\n    for index, name in enumerate(test_files, 1):\n        image_path = local_img / name\n        image = imread_unicode_with_retry(image_path)\n        if image is None:\n            raise FileNotFoundError(image_path)\n\n        # 3 detector 提案 (floor 0.08) → cons2of3_c26 放出\n        props = []\n        for det_name, model in det_models.items():\n            result = model.predict(\n                str(image_path), imgsz=YOLO_IMGSZ, conf=DET_FLOOR,\n                iou=PREDICT_IOU, agnostic_nms=True, retina_masks=True,\n                device=args.device, verbose=False)[0]\n            masks, confidences = result_masks(result)\n            for m, s in zip(masks, confidences):\n                if s >= DET_FLOOR and m.sum() >= MIN_AREA:\n                    props.append((det_name, m, float(s)))\n        emitted = emit_cons2of3_c26(props)\n        proposals = [m for m, _ in emitted]\n        emitted_conf = np.asarray([s for _, s in emitted], dtype=np.float32)\n\n        # 以降 prod 経路 verbatim: refine → threshold → ownership → RLE\n        boxes = [square_box(p) for p in proposals]\n        probabilities = refine_crops(refiner, image, boxes) if boxes else []\n        refined = []\n        for i, (prop, (x0, y0, x1, y1)) in enumerate(zip(proposals, boxes)):\n            mask = np.zeros_like(prop, dtype=bool)\n            mask[y0:y1, x0:x1] = probabilities[i] >= REFINER_THRESHOLD\n            refined.append(mask & prop)\n        final_masks = apply_ownership(\n            refined, emitted_conf,\n            rule="drop_contained_then_area_asc",\n            containment_threshold=DROP_THRESHOLD, min_area=MIN_AREA)\n        audit.append({"file_name": name, "proposals_all_dets": len(props),\n                      "consensus_emitted": len(emitted),\n                      "submitted": len(final_masks)})\n\n        stem = Path(name).stem\n        for i_m, mask in enumerate(final_masks, start=1):\n            encoded = mask_util.encode(\n                np.asfortranarray(mask.astype(np.uint8)))\n            rows.append({"filament_id": f"{stem}_{i_m}",\n                         "segmentation_rle": encoded["counts"].decode("utf-8")})\n        if index % 10 == 0 or index == len(test_files):\n            elapsed = time.perf_counter() - started\n            print(f"  {index}/{len(test_files)}  {elapsed:.0f}s  "\n                  f"残見積 {elapsed / index * (len(test_files) - index):.0f}s",\n                  flush=True)\n\n    total_s = time.perf_counter() - started\n    tag = "consensus"\n    submission = pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])\n    out_csv = args.out / f"submission_{tag}.csv"\n    submission.to_csv(out_csv, index=False, lineterminator="\\n")\n    efficiency = {"mode": "consensus_cons2of3_c26", "detectors": list(CONSENSUS_DETECTORS),\n                  "refiner": "edgeatt_all", "test_images": len(test_files),\n                  "total_wall_s": round(total_s, 1),\n                  "s_per_image": round(total_s / len(test_files), 3),\n                  "device": args.device, "threads": args.threads,\n                  "consensus": {"det_floor": DET_FLOOR, "iou": CONSENSUS_IOU,\n                                "support_conf": SUPPORT_CONF}}\n    (args.out / f"efficiency_{tag}.json").write_text(\n        json.dumps(efficiency, indent=2), encoding="utf-8")\n    (args.out / f"audit_{tag}.json").write_text(\n        json.dumps(audit, ensure_ascii=False, indent=1), encoding="utf-8")\n    sha = subprocess.run(["git", "rev-parse", "HEAD"], cwd=ROOT,\n                         capture_output=True, text=True).stdout.strip()\n    print(json.dumps({"submission": str(out_csv), "rows": len(submission),\n                      "images_with_predictions": sum(\n                          1 for a in audit if a["submitted"] > 0),\n                      "efficiency": efficiency, "code_sha": sha},\n                     ensure_ascii=False, indent=2))\n    return 0\n\n\ndef main() -> int:\n    # Each invocation stages its own inputs; no previous Drive/code generation\n    # can leave same-sized weights or same-named preprocessed images behind.\n    with tempfile.TemporaryDirectory(prefix="solar_consensus_") as staging:\n        return _run_submission(Path(staging))\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
script_path = REPRO / 'scratch' / 'run_test_submission_consensus.py'
script_path.write_text(CONSENSUS_SUB)
import ast; ast.parse(CONSENSUS_SUB)
print('consensus submission script written:', script_path)


In [ ]:
%%time
# [3/4] consensus 推論 (3 det + cluster + cons2of3_c26 → prod 後処理、GPU ~20-30min)
import subprocess, sys, torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, torch.cuda.get_device_name(0) if device == 'cuda' else '')
rc = subprocess.run([
    sys.executable, str(script_path),
    '--drive', str(DRIVE), '--device', device,
    '--out', '/content/repro_consensus'],
    capture_output=True, text=True)
print(rc.stdout[-4000:])
print('STDERR:', rc.stderr[-2000:])
assert rc.returncode == 0, 'consensus submission 生成失敗'


In [ ]:
# [4/4] 成果物を Drive に保存 + 検証 (提出はユーザー判断)
import json, shutil
import pandas as pd
OUT = Path('/content/repro_consensus')
DEST = V30 / 'artifacts' / 'consensus_v122'
DEST.mkdir(parents=True, exist_ok=True)
for f in OUT.iterdir():
    shutil.copy2(f, DEST / f.name)
    print('saved:', f.name)
df = pd.read_csv(DEST / 'submission_consensus.csv')
print('rows:', len(df), '| dup:', df.filament_id.duplicated().sum())
audit = json.loads((DEST / 'audit_consensus.json').read_text())
print('emitted/img mean:', sum(a['consensus_emitted'] for a in audit) / len(audit))
print('提出する場合は Kaggle UI で submission_consensus.csv をアップロード')
